# News Perspective Comparator
ทดสอบระบบเปรียบเทียบมุมมองข่าวจากหลายสำนัก

**Flow:** URL/Keyword → Jina Reader → World News API → Claude API → แสดงผล

## Step 1 — Install libraries

In [1]:
!pip install google-genai -q
from google import genai

## Step 2 — ใส่ API Keys

In [2]:
import os
WORLD_NEWS_API_KEY = os.environ.get("WORLD_NEWS_API_KEY", "")
GEMINI_API_KEY1 = os.environ.get("GEMINI_API_KEY1", "")
GEMINI_API_KEY2 = os.environ.get("GEMINI_API_KEY2", "")
GEMINI_API_KEY3 = os.environ.get("GEMINI_API_KEY3", "")
GEMINI_API_KEY4 = os.environ.get("GEMINI_API_KEY4", "")
GEMINI_API_KEY5 = os.environ.get("GEMINI_API_KEY5", "")
client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-3.8-flash"

## Step 3 — Jina Reader: แปลง URL → keyword

In [10]:
import requests

def extract_keyword_from_url(url: str) -> str:
    """
    ส่ง URL ให้ Jina Reader แปลงเป็น text
    แล้วให้ gemini สกัด keyword หลัก 1 ประโยค
    """
    # Jina Reader ดึง clean text จาก URL
    jina_url = f"https://r.jina.ai/{url}"
    res = requests.get(jina_url, timeout=10)
    article_text = res.text[:1000]
    print(article_text)

    # ให้ gemini สกัด keyword
    prompt = f"""Extract the main topic of this news article as a short search query
    - Maximum 3-4 words
    - English only
    - Keep only the most important proper nouns (person name, place, or event).
    - Do NOT include people's names unless they are the central figure.
    - Do NOT include outcomes or results.
    - Return ONLY the search query, nothing else.

Article text:
{article_text}"""

    response = client.models.generate_content(model=MODEL, contents=prompt)
    result = response.text
    keyword = response.text.strip()
    print(f"Extracted keyword: {keyword}")
    return keyword


# ทดสอบข่าว
test_url = "https://www.aljazeera.com/opinions/2026/10/2/washingtons-diplomacy-in-sudan-has-backfired"
keyword = extract_keyword_from_url(test_url)

{"data":null,"code":403,"name":"AbuseAlleviationError","status":40305,"message":"Anonymous access to domain www.aljazeera.com blocked until Sat Oct 03 2026 13:49:28 GMT+0000 (Coordinated Universal Time) due to previous abuse found on https://www.aljazeera.com/news/2020/12/9/uefa-to-open-investigation-over-racism-at-paris-football-match: DDoS attack suspected: Too many requests","readableMessage":"AbuseAlleviationError: Anonymous access to domain www.aljazeera.com blocked until Sat Oct 03 2026 13:49:28 GMT+0000 (Coordinated Universal Time) due to previous abuse found on https://www.aljazeera.com/news/2020/12/9/uefa-to-open-investigation-over-racism-at-paris-football-match: DDoS attack suspected: Too many requests"}
Extracted keyword: UEFA racism investigation


## Step 4 — World News API: ดึงข่าวหลายสำนัก

In [4]:
from datetime import datetime, timedelta
def fetch_news(keyword: str, api_key: str, max_results: int = 10, days_back: int = 7):
    """
    ดึงข่าวจาก World News API ตาม keyword
    คืน list ของบทความจากหลายสำนัก
    """
    url = "https://api.worldnewsapi.com/search-news"
    params = {
        "text": keyword,
        "language": "en",
        "number": max_results,
        "earliest-publish-date": (datetime.now() - timedelta(days=days_back)).strftime("%Y-%m-%d"),
        "api-key":  WORLD_NEWS_API_KEY,
    }

    res = requests.get(url, params=params, timeout=10)
    print("Status code:", res.status_code)
    print("Response:", res.text[:500])
    data = res.json()

    articles = []
    for item in data.get("news", []):
        articles.append({
            "source": item.get("source_country", "Unknown"),
            "title":  item.get("title", ""),
            "text":   item.get("text", "")[:800],  # ตัดให้ไม่ยาวเกิน
            "url":    item.get("url", ""),
            "sentiment": item.get("sentiment", 0),  # -1 ถึง 1
        })

    keywords_list = keyword.lower().split()
    articles = [
        a for a in articles
        if any(
            kw in a["title"].lower() or kw in a["text"].lower()
            for kw in keywords_list
        )
    ]

    seen_sources = set()
    unique_articles = []
    for a in articles:
        if a["source"] not in seen_sources:
            seen_sources.add(a["source"])
            unique_articles.append(a)
    articles = unique_articles

    print(f"After dedup: {len(articles)} unique sources")
    print(f"Found {len(articles)} articles")
    print(f"After filtering: {len(articles)} relevant articles")
    for a in articles:
      print(f"  - [{a['source']}] {a['title'][:60]} {a['text'][:200]}...")
    return articles


# ทดสอบ
articles = fetch_news(keyword, WORLD_NEWS_API_KEY, max_results=10, days_back=7)

Status code: 200
Response: {"offset":0,"number":10,"available":21,"news":[{"id":471307208,"title":"Late-night talks aimed at resolving Garvaghy Road stand-off end without agreement","text":"Sign in. It’s quick, free and it’s up to you.\n\nAn account is an optional way to support the work we do. Find out more.\n\nBest of the Box\n\nInvestigates\n\nInvestigates\n\nMoney Diaries\n\nThe Journal TV\n\nClimate Crisis\n\nCost of Living\nRoad Safety\n\nNewsletters\n\nTemperature Check\nInside the Newsroom\nThe Journal Investigate
After dedup: 2 unique sources
Found 2 articles
After filtering: 2 relevant articles
  - [ie] 'It's definitely been testing, psychologically, 100%' - Irel League of Ireland

Horse Racing

TV Listings

GAA Fixtures
The Video Review

Sport meets news, current affairs, society & pop culture

Rugby Weekly Extra

Dive into all the news and analysis 3 times a...
  - [gb] FAI to investigate alleged racist abuse of Swansea City star The Football Association of Ireland has conf

## Step 5 — Gemini API: เปรียบเทียบมุมมองแต่ละสำนัก

In [5]:
import json

def compare_perspectives(articles: list) -> list:
    """
    ส่งบทความทั้งหมดให้ Gemini วิเคราะห์
    คืน list ของ {source, summary, angle, sentiment_label}
    """
    if not articles:
        return []

    # รวมบทความทั้งหมดเป็น prompt เดียว
    articles_text = ""
    for i, a in enumerate(articles, 1):
        articles_text += f"""
--- Article {i} ---
Source: {a['source']}
Title: {a['title']}
Content: {a['text']}
"""

    prompt = f"""You are analyzing news articles about the same topic from different sources.
Topic: "{keyword}"

{articles_text}

Analyze ONLY articles that are directly about this topic. Skip any that are clearly unrelated.
For each relevant article, provide a JSON array with objects containing:
- "source": the source name
- "title": original title
- "summary": 2-sentence summary in English
- "angle": what angle or framing this source emphasizes (1 sentence)
- "sentiment": "positive", "neutral", or "negative"
- "unique_point": one thing this source mentions that others don't

Return ONLY valid JSON array, no other text."""

    response = client.models.generate_content(model=MODEL, contents=prompt)
    raw = response.text.strip().removeprefix("```json").removesuffix("```").strip()
    results = json.loads(raw)
    return results


# ทดสอบ
results = compare_perspectives(articles)
print(json.dumps(results, indent=2))

[
  {
    "source": "gb",
    "title": "FAI to investigate alleged racist abuse of Swansea City star during Israel clash",
    "summary": "The Football Association of Ireland is launching an investigation into alleged racist chants directed at striker Adam Idah during a Nations League match against Israel. UEFA will not be investigating the incident because the necessary reporting protocols were not followed during the match.",
    "angle": "The article emphasizes the procedural failures that prevented UEFA's direct involvement in the investigation.",
    "sentiment": "neutral",
    "unique_point": "It specifies that the investigation is being led solely by the FAI because the formal reporting protocols were not triggered by players or officials on the night of the match."
  }
]


## Step 6 — แสดงผลสวยๆ ใน Colab

In [6]:
from IPython.display import display, HTML

def sentiment_color(sentiment: str) -> str:
    return {
        "positive": "#d4edda",
        "negative": "#f8d7da",
        "neutral":  "#fff3cd",
    }.get(sentiment.lower(), "#f0f0f0")


def display_results(results: list, keyword: str):
    html = f"""<h2>Topic: {keyword}</h2>"""
    html += f"""<p>Found perspectives from {len(results)} sources</p>"""

    for r in results:
        color = sentiment_color(r.get("sentiment", "neutral"))
        url   = next((a["url"] for a in articles if a["title"] == r.get("title")), "#")
        html += f"""
        <div style="border:1px solid #ddd; border-radius:8px;
                    padding:16px; margin:12px 0; background:{color}; color: #000;">
          <h3 style="margin:0 0 4px; color: #000;">{r.get('source','?')}</h3>
          <p style="margin:0 0 8px"><a href="{url}" target="_blank" style="color: #000;">{r.get('title','')}</a></p>
          <p><b>Summary:</b> {r.get('summary','')}</p>
          <p><b>Angle:</b> {r.get('angle','')}</p>
          <p><b>Unique point:</b> {r.get('unique_point','')}</p>
          <span style="background:#fff; border-radius:4px; padding:2px 8px; font-size:12px">
            Sentiment: {r.get('sentiment','?')}
          </span>
        </div>"""

    display(HTML(html))


display_results(results, keyword)

## Step 7 — รันทุกอย่างในครั้งเดียว (รวม pipeline)

In [7]:
def clean_keyword(user_input: str) -> str:
    prompt = f"""Convert this text into a short news search query (maximum 4 words, English only).
Return ONLY the search query, nothing else.

Text: {user_input}"""

    response = client.models.generate_content(model=MODEL, contents=prompt)
    return response.text.strip()

In [8]:
def run_pipeline(input_text: str, is_url: bool = False):
    """
    input_text: URL หรือ keyword
    is_url: True ถ้าเป็น URL, False ถ้าเป็น keyword
    """
    print("Step 1: Getting keyword...")
    if is_url:
        keyword = extract_keyword_from_url(input_text)
    else:
        keyword = clean_keyword(input_text)
        print(f"Using keyword: {keyword}")

    print("\nStep 2: Fetching news...")
    articles = fetch_news(keyword, WORLD_NEWS_API_KEY, max_results=10)

    if not articles:
        print("No articles found. Try a different keyword.")
        return

    print("\nStep 3: Analyzing perspectives...")
    results = compare_perspectives(articles)

    print("\nStep 4: Displaying results...")
    display_results(results, keyword)


# ทดสอบด้วย keyword
run_pipeline("""Brydon Carse Pakistan Test""", is_url=False)

# หรือทดสอบด้วย URL
# run_pipeline("https://www.bbc.com/sport/football/videos/cx2zngdln6go", is_url=True)

Step 1: Getting keyword...
Using keyword: Brydon Carse Pakistan Test

Step 2: Fetching news...
Status code: 200
Response: {"offset":0,"number":10,"available":12,"news":[{"id":471482054,"title":"England's Carse charged by cricket body over nightclub incident","text":"England bowler Brydon Carse has been charged with breaching the England and Wales Cricket Board's (ECB) conduct regulations following a nightclub incident.\n\nThe charge, which follows an investigation by England's Cricket Regulator, refers to events that took place on August 23 when Carse was handcuffed by police officers after a celebratory night out 
After dedup: 3 unique sources
Found 3 articles
After filtering: 3 relevant articles
  - [za] England's Carse charged by cricket body over nightclub incid England bowler Brydon Carse has been charged with breaching the England and Wales Cricket Board's (ECB) conduct regulations following a nightclub incident.

The charge, which follows an investigation ...
  - [in] England pa